In [ ]:
%reset -f
import mfem.ser as mfem
import numpy as np
from glvis import glvis

### Problema de Darcy: formulación mixta
$$\begin{array}{c}
\begin{array}{r}
 {\bf u} + \nabla p = {\bf f} \\
\nabla \cdot{\bf u} = g \end{array} \quad\text{en }\Omega \\
\begin{array}{r}
p = p_0 \end{array} \quad\text{en }\partial\Omega 
\end{array}
$$
para ${\bf f}={\bf 0}$, $g=e^x \sin y\cos z$ y $p_0=g$. La solución es $p=g$ y ${\bf u}=-\nabla g$

#### Definición de funciones

In [ ]:
# Función auxiliar g 
def pFunc_exact(x):
    return np.exp(x[0])*np.sin(x[1])*np.cos(x[2])

## Soluciones exactas (para luego comparar)
# Velocidad exacta u = -\nabla g
class uFunc_ex(mfem.VectorPyCoefficient): 
    def EvalValue(self, x):
        ret = [- np.exp(x[0])*np.sin(x[1])*np.cos(x[2]),
               - np.exp(x[0])*np.cos(x[1])*np.cos(x[2]),
                np.exp(x[0])*np.sin(x[1])*np.sin(x[2])]
        return ret

# Presión exacta p=g
class pFunc_ex(mfem.PyCoefficient):
    def EvalValue(self, x):
        return pFunc_exact(x)


# f (segundo miembro  de u+\nabla p =f)
class fFunc(mfem.VectorPyCoefficient):
    def EvalValue(self, x):
        return [0., 0., 0.]

# g (segundo miembro de -\nabla \cdot u = -g) 
class gFunc(mfem.PyCoefficient):
    def EvalValue(self, x):
        return -pFunc_exact(x)

# p_0
class f_natural(mfem.PyCoefficient):
    def EvalValue(self, x):
        return -pFunc_exact(x)

### Lectura de malla

In [ ]:
meshfile = 'mallas/fichera.mesh'

mesh = mfem.Mesh(meshfile, 1, 1)
dim = mesh.Dimension()

ref_levels = 3
for x in range(ref_levels):
    mesh.UniformRefinement()

#### Espacios 
* Raviart-Thomas: velocidad
* L2: presión (orden 1)

In [ ]:
order = 1
hdiv = mfem.RT_FECollection(order, dim)
l2 = mfem.L2_FECollection(order, dim)

R_space = mfem.FiniteElementSpace(mesh, hdiv)
W_space = mfem.FiniteElementSpace(mesh, l2)

#### Dimensiones de los espacios y configuración de los bloques
La solución vendrá dada un vector que auna la velocidad y la presión en un `BlockVector`

In [ ]:
dimR = R_space.GetVSize()
dimW = W_space.GetVSize()

# dimensiones
block_offsets = mfem.intArray([0, dimR, dimW])
block_offsets.PartialSum()

# vectores
x = mfem.BlockVector(block_offsets)
rhs = mfem.BlockVector(block_offsets)

### Formulación variacional
Hallar $({\bf u},p)\in H({\rm div},\Omega) \times L^2(\Omega)$

$$\begin{array}{c}
\displaystyle \int_\Omega {\bf u}\cdot {\bf v} - \int_\Omega p(\nabla\cdot {\bf v}) = 
\int_\Omega {\bf f}\cdot {\bf v} - \int_{\partial \Omega} p_0({\bf v}\cdot \vec{n})
\quad \forall {\bf v} \in H({\rm div}, \Omega), \\
-\displaystyle \int_\Omega (\nabla \cdot{\bf u})q = -\int_\Omega g q,\quad \forall q \in L^2(\Omega)
\end{array}$$
con ${\bf f},g \in L^2(\Omega)$, $p_0\in H^{\frac 12}(\partial\Omega)$.

##### Matricialmente:
$$\begin{bmatrix} M & B^T \\ B & 0\end{bmatrix} \begin{bmatrix} U \\ P \end{bmatrix} = 
\begin{bmatrix} F \\ G \end{bmatrix} $$
con $$\begin{array}{l}
M \longrightarrow \displaystyle \int_\Omega {\bf u}\cdot {\bf v} \\
B \longrightarrow -\displaystyle \int_\Omega (\nabla \cdot {\bf u} ) q \Rightarrow B^T \longrightarrow - \displaystyle \int_\Omega p(\nabla \cdot {\bf v})
\end{array}$$

Coeficientes del problema

In [ ]:
fcoeff = fFunc(dim) # f
fnatcoeff = f_natural() # -p_0
gcoeff = gFunc() #(-g)

Formas bilineales de la matriz

In [ ]:
mVarf = mfem.BilinearForm(R_space)
mVarf.AddDomainIntegrator(mfem.VectorFEMassIntegrator())
mVarf.Assemble()
mVarf.Finalize()

bVarf = mfem.MixedBilinearForm(R_space, W_space)
bVarf.AddDomainIntegrator(mfem.VectorFEDivergenceIntegrator(mfem.ConstantCoefficient(-1.)))
bVarf.Assemble()
bVarf.Finalize()

Segundos miembros

In [ ]:
fform = mfem.LinearForm()
# enlazamos con el vector rhs
fform.Update(R_space, rhs.GetBlock(0), 0)
fform.AddDomainIntegrator(mfem.VectorFEDomainLFIntegrator(fcoeff))
fform.AddBoundaryIntegrator(mfem.VectorFEBoundaryFluxLFIntegrator(fnatcoeff))
fform.Assemble()

gform = mfem.LinearForm()
gform.Update(W_space, rhs.GetBlock(1), 0)
gform.AddDomainIntegrator(mfem.DomainLFIntegrator(gcoeff))
gform.Assemble()

### Matriz del sistema

In [ ]:
darcyOp = mfem.BlockOperator(block_offsets)

M = mVarf.SpMat()
B = bVarf.SpMat()
Bt = mfem.TransposeOperator(B)

darcyOp.SetBlock(0, 0, M)
darcyOp.SetBlock(0, 1, Bt)
darcyOp.SetBlock(1, 0, B)

#### Resolución

In [ ]:
solver = mfem.MINRESSolver()
solver.SetAbsTol(0.)
solver.SetRelTol(1.e-6)
solver.SetMaxIter(1000)
solver.SetOperator(darcyOp)
solver.SetPrintLevel(0)
x.Assign(0.0)
solver.Mult(rhs, x)


if solver.GetConverged():
    print("MINRES convergió en " + str(solver.GetNumIterations()) +
          " iteraciones, con residuo " + "{:g}".format(solver.GetFinalNorm()))
else:
    print("MINRES no ha convergido tras " + str(solver.GetNumIterations()) +
          " iteraciones. Residuo:" + "{:g}".format(solver.GetFinalNorm()))

#### Visualización de resultados
Asociamos cada porción del vector solución `x` a una `GridFunction` en el espacio oportuno mediante `MakeRef`

In [ ]:
u = mfem.GridFunction()
p = mfem.GridFunction()
u.MakeRef(R_space, x.GetBlock(0), 0)
p.MakeRef(W_space, x.GetBlock(1), 0)

Velocidad

In [ ]:
glvis((mesh,u))

Presión

In [ ]:
glvis((mesh,p))

#### Cálculo del error
Comparación con solución exacta

`irs` hace referencia a la fórmula de cuadratura usada (dada por el orden `order_quad`) para calcular las normas

In [ ]:
# Coeficientes de la solución exacta
ucoeff = uFunc_ex(dim)
pcoeff = pFunc_ex()
order_quad = max(2, 2*order+1)

irs = [mfem.IntRules.Get(i, order_quad)
       for i in range(mfem.Geometry.NumGeom)]

norm_p = mfem.ComputeLpNorm(2, pcoeff, mesh, irs)
norm_u = mfem.ComputeLpNorm(2, ucoeff, mesh, irs)
err_u = u.ComputeL2Error(ucoeff, irs)
err_p = p.ComputeL2Error(pcoeff, irs)

print("|| u_h - u_ex || / || u_ex || = " + "{:g}".format(err_u / norm_u))
print("|| p_h - p_ex || / || p_ex || = " + "{:g}".format(err_p / norm_p))

